In [1]:
"""
Importing Libraries
These will be used for numerical operations, plotting, animating, timng, and priority queue operations
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import time
import heapq

# Formalism

The state is represented by the coordinates and directions of the BeeBot. The map is not included in the state and only serves as a reference to check the BeeBots position on the map and weather it has reached its goal.

<u>BeeBot</u> (The State)
* Represented by a 1D array [bee_row, bee_col, bee_direction]
* bee_direction index corresponds to the following: 0 = North, 1 = East, 2 = South, 3 = West
* Initial State is randomly initialised, whereby map[bee_row][bee_col] == 0
* Goal state reached when map[bee_row][bee_col] == 2

<u>Map</u>
* represented by 2D array (m by n dimensions)
* Empty cell = 0
* Obstacles = -1
* BeeBot = 1
* Goal Cell = 2

<u>Conditions</u>
* 0 <= bee_row <= m
* 0 <= bee_col <= n
* 0 <= bee_direction <= 3
* map[bee_row][bee_col] != -1 (BeeBot cannot be on an obstacle labeled -1)

# Defining List of Possible Actions

## List of Actions Function

* This function sequences the possible actions based on the current direction that the BeeBot is facing
* The BeeBot has four actions: move forwards, move backwards, turn 90 degrees left, turn 90 degrees right
* The directions from the BeeBots perspective may not necessarily align with the cardinal directions on the map. For example, if BeeBot is currently facing East and it was to turn and move to its right, it would be moving further down (South) of the map rather than actually moving to the right (East) of the map.
* For the purpose of DFS, the sequence of actions are: 1) step back, 2) turn left, 3) turn right 4) step forward. This is to prioritise the step-forward action since DFS is Last-In-First-Out.

In [3]:
def ListOfActions(bee, action):
    x, y, direction = bee # obtaining the beebots current position and direction

    # define the movent deltas for each direction
    #these represent changes in x and y for each of the four cardinal directions

    movement_deltas = {
        0: (-1, 0), # Moving North decreases x by 1
        1: (0, 1), # Moving East increases y by 1
        2: (1, 0), # Moving SOuth increases x by 1
        3: (0, -1) # Moving West decreases y by 1
    }
    # Handle different actions
    if action == "move_forward":
        dx, dy = movement_deltas[direction]
        new_x, new_y = x + dx, y + dy
    elif action == "move_backward":
        dx, dy = movement_deltas[direction]
        new_x, new_y = x - dx, y - dy
    elif action == "rotate_left":
        new_direction = (direction - 1) % 4
        return (x, y, new_direction)
    elif action == "rotate_right":
        new_direction = (direction + 1) % 4
        return (x, y, new_direction)
    else:
        # if the move is invalid, return the current state unchanged
        return bee 

    # check if BeeBot is in bounds and not on an obstacle
    if 0 <= new_x < size and 0 <= new_y < size and map_matrix[new_x, new_y] != -1:
        return(new_x, new_y, direction)
    else:
        return bee
        

# Constructing Map and Initialising BeeBot

In [ ]:
size = int(input('Enter Map Size: '))

map_matrix = np.zeros((size, size), dtype = int) # Creating MxN matrix filled with zeros

# Creating the Obstacles

num_neg_ones = (size*size)*0.2 # define number of obstacles (-1 values) (20% of the map)
neg_one_indices = [] # empty array to place the coordinates of the obstacles

# ensure that obstacles do not overwrite the initial BeeBotor goal positions
while len(neg_one_indices) < num_neg_ones:
    i, j = np.random.randint(0, size, 2) # randomly placing the obstacles on the map
    neg_one_indices.append((i,j))

# PLace the -1 values
for i, j in neg_one_indices:
    map_matrix[i, j] = -1 # overwriting the matrix of 0's with -1 values

while True:
    bee_x, bee_y = np.random.randint(0, size, 2):
    if map_matrix[bee_x, bee_y] == 0:
        break

# Randomly selecting the starting orientation of the BeeBot
bee_direction = np.random.randint (0, 4)

# Combining the Position and Orientation of the BeeBot
bee = (bee_x, bee_y, bee_direction)

# Randomly selecting where the goal is placed
while True:
    goal_x, goal_y = np.random.randint(0, size, 2)
    if map_matrix[goal_x, goal_y] == 0 and (goal_x, goal_y) != (bee_x, bee_y):
        break
